# Laboratorio 6 - Visión Por Computadora

Autores:

- Nelson García
- Joaquín Puente
- Diego Linares

# Task 1

1. Un desarrollador junior de su equipo sugiere: "Para detectar con mayor precisión las texturas de las
hojas enfermas, deberíamos construir una red secuencial clásica (tipo VGG) pero de 150 capas. Más
profundo siempre es mejor". Como líder técnico, explíquele argumentativamente por qué esta red
fracasará estrepitosamente en el entrenamiento (mencionando el fenómeno de degradación y el
desvanecimiento del gradiente). Luego, justifique cómo la adición estructural de las conexiones
residuales (𝐹(𝑥) + 𝑥) de ResNet rescata el proyecto, haciendo viable entrenar redes ultra-profundas
sin colapsar.

R//

La propuesta de construir una red secuencial clásica tipo VGG con 150 capas parte de una intuición incompleta. Es cierto que VGG mostró que aumentar la profundidad puede mejorar la capacidad de representación, pero existe un límite práctico y matemático. En redes muy profundas aparecen dos problemas centrales. El primero es el desvanecimiento del gradiente, donde durante la retropropagación los gradientes se vuelven cada vez más pequeños al atravesar muchas capas, por lo que las capas tempranas casi dejan de aprender. El segundo es la degradación de la precisión, que significa que al seguir agregando capas el error de entrenamiento aumenta en vez de disminuir. El material de clase lo resume de forma directa al indicar que una mayor profundidad puede producir mayor error de entrenamiento y que esto no debe confundirse con sobreajuste.

En una arquitectura puramente secuencial, cada capa debe aprender una transformación completa sobre la salida anterior. En teoría esto parece más expresivo, pero en la práctica vuelve muy inestable la optimización. Si una parte de la red adicional no aporta valor, el modelo no tiene un mecanismo simple para comportarse como identidad y dejar pasar la información sin distorsionarla. Por eso una VGG extrema de 150 capas tendería a entrenar lentamente, estancarse o incluso rendir peor que una versión mucho más corta, aun teniendo más capacidad nominal.

ResNet rescata el proyecto porque cambia la pregunta que aprende cada bloque. En lugar de obligar a la red a aproximar directamente una transformación completa H(x), el bloque residual aprende solo el residuo F(x), de modo que la salida final se expresa como H(x) = F(x) + x. Esta suma con la identidad no es un detalle cosmético, sino una modificación estructural profunda. En la retropropagación aparece un término adicional igual a 1 en la derivada con respecto a la entrada del bloque, lo cual crea una ruta directa para el flujo del gradiente y reduce el desvanecimiento. En otras palabras, aunque la rama convolucional aprenda poco o se vuelva difícil de optimizar, la señal todavía puede propagarse por el atajo de identidad.

Desde la perspectiva del liderazgo técnico, la conclusión es clara. No conviene apostar el proyecto AgriTech a una VGG de 150 capas solo por ser más profunda. Esa decisión elevaría el riesgo de entrenamiento fallido, pérdida de tiempo de GPU y retrasos de entrega. En cambio, una ResNet convierte la profundidad en una ventaja utilizable, porque preserva el flujo de información y de gradientes, reduce el colapso de optimización y vuelve factible entrenar modelos ultra profundos para capturar texturas finas en hojas enfermas sin destruir la estabilidad del proceso.


2. Las enfermedades en las hojas de mango son visualmente heterogéneas: La Antracnosis se
presenta como puntos negros diminutos, mientras que el Moho Polvoriento cubre áreas enormes de
la hoja. Analice cómo la topología en paralelo del módulo Inception (usando filtros 3x3 y 5x5
simultáneamente) es ideal para este problema biológico en particular. Además, desde una
perspectiva de costos de infraestructura (uso de GPUs en AWS o Google Cloud), explique cómo la
inserción estratégica de convoluciones de 1x1 evita la explosión de la dimensionalidad y salva el
presupuesto mensual de la startup.

R//

El problema biológico de las hojas de mango exige una arquitectura que vea patrones en múltiples escalas espaciales al mismo tiempo. La Antracnosis puede manifestarse como puntos negros diminutos, lo que requiere filtros pequeños capaces de capturar detalles locales y texturas finas. En cambio, el Moho Polvoriento cubre regiones más amplias de la hoja, por lo que conviene usar filtros mayores que integren contexto espacial más global. El material de clase justamente explica que los filtros pequeños como 3x3 capturan detalles finos, mientras que filtros más grandes como 5x5 o 7x7 capturan contexto global.

Aquí es donde el módulo Inception resulta especialmente adecuado. Su topología en paralelo evita obligar al diseñador a escoger un único tamaño de filtro antes de entrenar. En lugar de eso, procesa la misma entrada por varias ramas simultáneas, por ejemplo con 3x3 y 5x5, y luego concatena los mapas de características obtenidos. Así, la red aprende de manera conjunta rasgos pequeños y patrones extensos. Para un dominio tan heterogéneo como la fitopatología foliar, esto es una ventaja decisiva, porque una sola imagen puede contener tanto lesiones puntuales como zonas infectadas de gran cobertura. El módulo Inception fue precisamente concebido como una extracción a múltiples escalas espaciales.

Sin embargo, esa riqueza multiescala tiene un peligro operativo. Si cada rama procesa todos los canales de entrada con convoluciones costosas y luego se concatenan todas las salidas, la dimensionalidad puede crecer rápidamente. Esto incrementa parámetros, memoria intermedia y FLOPs, lo cual se traduce en tiempos de entrenamiento más altos y facturas más caras en GPUs de AWS o Google Cloud. El material lo describe como una explosión combinatoria de dimensiones y advierte que la concatenación puede generar mapas inmanejables.

La solución elegante es insertar convoluciones de 1x1 antes de las ramas 3x3 y 5x5. Matemáticamente, estas capas actúan como proyectores lineales sobre la dimensión de canales. No cambian la resolución espacial, pero sí reducen C, es decir, la cantidad de canales que entrará a las convoluciones posteriores. Como el costo computacional de la convolución crece con N, C, W y H, reducir canales antes de aplicar filtros más grandes disminuye de manera importante las operaciones requeridas. Por eso el material destaca que las convoluciones 1x1 permiten reducción preventiva de dimensionalidad y mantienen alta expresividad con bajo costo.

Traducido a decisiones de negocio, esto significa que Inception no solo mejora la sensibilidad del modelo ante síntomas visuales muy distintos, sino que también protege el presupuesto mensual de la startup. Menos FLOPs implican menor tiempo por época, menor uso de memoria y posibilidad de entrenar lotes razonables sin subir a instancias más costosas. En un entorno startup, donde cada hora de GPU cuenta, usar 1x1 antes de 3x3 y 5x5 no es una optimización menor, sino una medida financiera y técnica al mismo tiempo. Por ello, para AgriTech, Inception ofrece un balance muy conveniente entre diversidad espacial y eficiencia computacional.

3. El cliente final (el agricultor) usará la aplicación en un teléfono Android de gama baja en medio del
campo, sin conexión a internet. Sabemos que MobileNet logra esta eficiencia gracias a la Depthwise
Separable Convolution. Describa brevemente cómo esta convolución divide el trabajo (filtrado
espacial vs. combinación de canales). Sin embargo, en ingeniería no hay soluciones mágicas, todo tiene un precio: ¿Qué capacidad matemática o expresividad de la red se está sacrificando al separar
estos dos pasos en comparación con una convolución estándar? Analice críticamente por qué, como
director del proyecto, usted acepta pagar ese "costo" en este escenario comercial.

R//



# Task 2

## Instalación y dependencias

In [ ]:
import os
import time
import copy
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

from sklearn.metrics import f1_score, classification_report, confusion_matrix
import seaborn as sns

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")

## 1. Descarga del dataset y división Train/Val/Test (70/15/15) con Data Augmentation

In [ ]:
import kagglehub

# Descargar el dataset de enfermedades de hojas de mango
path = kagglehub.dataset_download("aryashah2k/mango-leaf-disease-dataset")
print(f"Dataset descargado en: {path}")

# Buscar la carpeta raíz que contiene las subcarpetas de clases
for root, dirs, files in os.walk(path):
    if dirs and any(not d.startswith('.') for d in dirs):
        # Verificar si las subcarpetas contienen imágenes
        sample_dir = os.path.join(root, [d for d in dirs if not d.startswith('.')][0])
        if any(f.lower().endswith(('.jpg', '.jpeg', '.png')) for f in os.listdir(sample_dir)):
            DATA_ROOT = root
            break

print(f"Directorio raíz del dataset: {DATA_ROOT}")
print(f"Clases encontradas: {sorted(os.listdir(DATA_ROOT))}")
print(f"Número de clases: {len([d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d))])}")

In [ ]:
# Definir transformaciones
# Para entrenamiento: Data Augmentation (rotaciones, flips, recortes)
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# Para validación y prueba: solo resize y normalización
eval_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# InceptionV3 requiere entrada de 299x299
train_transform_inception = transforms.Compose([
    transforms.Resize((320, 320)),
    transforms.RandomResizedCrop(299, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

eval_transform_inception = transforms.Compose([
    transforms.Resize((320, 320)),
    transforms.CenterCrop(299),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [ ]:
# Cargar dataset completo (sin transformación aún, solo para obtener índices)
full_dataset = datasets.ImageFolder(DATA_ROOT)
class_names = full_dataset.classes
num_classes = len(class_names)

print(f"Total de imágenes: {len(full_dataset)}")
print(f"Clases ({num_classes}): {class_names}")

# Contar imágenes por clase
from collections import Counter
label_counts = Counter([label for _, label in full_dataset.samples])
for cls_idx, cls_name in enumerate(class_names):
    print(f"  {cls_name}: {label_counts[cls_idx]} imágenes")

In [ ]:
# División estratificada: 70% train, 15% val, 15% test
from sklearn.model_selection import train_test_split

targets = [s[1] for s in full_dataset.samples]
indices = list(range(len(full_dataset)))

# Primera división: 70% train, 30% temp
train_idx, temp_idx = train_test_split(
    indices, test_size=0.30, random_state=42, stratify=targets
)

# Segunda división: 50/50 del 30% → 15% val, 15% test
temp_targets = [targets[i] for i in temp_idx]
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50, random_state=42, stratify=temp_targets
)

print(f"Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")
print(f"Proporciones: {len(train_idx)/len(indices):.2f} / {len(val_idx)/len(indices):.2f} / {len(test_idx)/len(indices):.2f}")

In [ ]:
def get_dataloaders(train_tf, eval_tf, batch_size=32):
    """Crea DataLoaders aplicando las transformaciones correspondientes."""
    # Datasets con transformaciones específicas
    train_ds = datasets.ImageFolder(DATA_ROOT, transform=train_tf)
    val_ds = datasets.ImageFolder(DATA_ROOT, transform=eval_tf)
    test_ds = datasets.ImageFolder(DATA_ROOT, transform=eval_tf)

    # Subsets con los índices de la división
    train_subset = Subset(train_ds, train_idx)
    val_subset = Subset(val_ds, val_idx)
    test_subset = Subset(test_ds, test_idx)

    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_subset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    return train_loader, val_loader, test_loader

# Verificar una muestra
train_loader, val_loader, test_loader = get_dataloaders(train_transform, eval_transform)
images, labels = next(iter(train_loader))
print(f"Batch shape: {images.shape}, Labels: {labels[:8]}")

## 2. Modelos pre-entrenados con Transfer Learning

Se cargan ResNet50, InceptionV3 y MobileNetV2 pre-entrenados en ImageNet. Se congelan las capas base y se reemplaza únicamente el cabezal de clasificación para 8 clases.

In [ ]:
def build_resnet50(num_classes):
    """ResNet50 con capas base congeladas y nuevo cabezal."""
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    # Congelar todas las capas base
    for param in model.parameters():
        param.requires_grad = False
    # Reemplazar el cabezal de clasificación
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, num_classes)
    )
    return model

def build_inceptionv3(num_classes):
    """InceptionV3 con capas base congeladas y nuevo cabezal."""
    model = models.inception_v3(weights=models.Inception_V3_Weights.IMAGENET1K_V1)
    # Congelar todas las capas base
    for param in model.parameters():
        param.requires_grad = False
    # Reemplazar cabezal principal
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, num_classes)
    )
    # Reemplazar cabezal auxiliar
    in_features_aux = model.AuxLogits.fc.in_features
    model.AuxLogits.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features_aux, num_classes)
    )
    return model

def build_mobilenetv2(num_classes):
    """MobileNetV2 con capas base congeladas y nuevo cabezal."""
    model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
    # Congelar todas las capas base
    for param in model.parameters():
        param.requires_grad = False
    # Reemplazar el clasificador
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, num_classes)
    )
    return model

# Construir los 3 modelos
model_resnet = build_resnet50(num_classes).to(device)
model_inception = build_inceptionv3(num_classes).to(device)
model_mobilenet = build_mobilenetv2(num_classes).to(device)

# Verificar parámetros entrenables
for name, model in [("ResNet50", model_resnet), ("InceptionV3", model_inception), ("MobileNetV2", model_mobilenet)]:
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"{name}: {trainable:,} entrenables / {total:,} totales ({100*trainable/total:.1f}%)")

## 3. Entrenamiento con Cross-Entropy, Adam y Early Stopping

In [ ]:
def train_model(model, train_loader, val_loader, model_name,
                max_epochs=20, patience=5, lr=1e-3):
    """
    Entrena un modelo con Cross-Entropy Loss, Adam optimizer y Early Stopping.
    Retorna el modelo con los mejores pesos y el historial de entrenamiento.
    """
    criterion = nn.CrossEntropyLoss()
    # Solo optimizar parámetros que requieren gradiente
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_loss = float('inf')
    best_model_wts = copy.deepcopy(model.state_dict())
    epochs_no_improve = 0

    for epoch in range(max_epochs):
        # --- Entrenamiento ---
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()

            # InceptionV3 retorna salida principal + auxiliar en modo train
            if model_name == "InceptionV3" and model.training:
                outputs, aux_outputs = model(images)
                loss = criterion(outputs, labels) + 0.4 * criterion(aux_outputs, labels)
            else:
                outputs = model(images)
                loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / total
        train_acc = correct / total

        # --- Validación ---
        model.eval()
        running_loss, correct, total = 0.0, 0, 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                running_loss += loss.item() * images.size(0)
                _, preds = torch.max(outputs, 1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)

        val_loss = running_loss / total
        val_acc = correct / total

        scheduler.step(val_loss)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        print(f"  Epoch {epoch+1}/{max_epochs} | "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

        # Early Stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"  Early stopping en epoch {epoch+1}")
                break

    model.load_state_dict(best_model_wts)
    return model, history

In [ ]:
# ==================== Entrenar ResNet50 ====================
print("=" * 60)
print("Entrenando ResNet50...")
print("=" * 60)
train_loader, val_loader, test_loader = get_dataloaders(train_transform, eval_transform)
model_resnet, history_resnet = train_model(
    model_resnet, train_loader, val_loader, "ResNet50", max_epochs=20, patience=5
)

In [ ]:
# ==================== Entrenar InceptionV3 ====================
print("=" * 60)
print("Entrenando InceptionV3...")
print("=" * 60)
train_loader_inc, val_loader_inc, test_loader_inc = get_dataloaders(
    train_transform_inception, eval_transform_inception
)
model_inception, history_inception = train_model(
    model_inception, train_loader_inc, val_loader_inc, "InceptionV3", max_epochs=20, patience=5
)

In [ ]:
# ==================== Entrenar MobileNetV2 ====================
print("=" * 60)
print("Entrenando MobileNetV2...")
print("=" * 60)
train_loader, val_loader, test_loader = get_dataloaders(train_transform, eval_transform)
model_mobilenet, history_mobilenet = train_model(
    model_mobilenet, train_loader, val_loader, "MobileNetV2", max_epochs=20, patience=5
)

### Curvas de entrenamiento

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 12))

for i, (name, hist) in enumerate([
    ("ResNet50", history_resnet),
    ("InceptionV3", history_inception),
    ("MobileNetV2", history_mobilenet)
]):
    # Loss
    axes[i, 0].plot(hist['train_loss'], label='Train')
    axes[i, 0].plot(hist['val_loss'], label='Val')
    axes[i, 0].set_title(f'{name} - Loss')
    axes[i, 0].set_xlabel('Epoch')
    axes[i, 0].set_ylabel('Loss')
    axes[i, 0].legend()
    axes[i, 0].grid(True)

    # Accuracy
    axes[i, 1].plot(hist['train_acc'], label='Train')
    axes[i, 1].plot(hist['val_acc'], label='Val')
    axes[i, 1].set_title(f'{name} - Accuracy')
    axes[i, 1].set_xlabel('Epoch')
    axes[i, 1].set_ylabel('Accuracy')
    axes[i, 1].legend()
    axes[i, 1].grid(True)

plt.tight_layout()
plt.show()

## 4. Evaluación y métricas comparativas

Para cada modelo se registra:
- **Accuracy** en test
- **F1-Score (Macro)** en test
- **Tamaño del modelo** en MB (.pth)
- **Tiempo de inferencia** promedio por imagen (ms, sobre 100 imágenes)

In [ ]:
def evaluate_model(model, test_loader, model_name, save_path):
    """Evalúa un modelo y retorna todas las métricas requeridas."""
    model.eval()
    all_preds = []
    all_labels = []

    # --- Accuracy y F1-Score ---
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = np.mean(np.array(all_preds) == np.array(all_labels))
    f1_macro = f1_score(all_labels, all_preds, average='macro')

    # --- Tamaño del modelo en disco ---
    torch.save(model.state_dict(), save_path)
    model_size_mb = os.path.getsize(save_path) / (1024 * 1024)

    # --- Tiempo de inferencia (promedio sobre 100 imágenes) ---
    # Obtener 100 imágenes del test set
    test_images = []
    for images, _ in test_loader:
        for img in images:
            test_images.append(img.unsqueeze(0).to(device))
            if len(test_images) >= 100:
                break
        if len(test_images) >= 100:
            break

    # Warmup
    with torch.no_grad():
        for _ in range(10):
            _ = model(test_images[0])

    # Medir tiempo
    if device.type == 'cuda':
        torch.cuda.synchronize()

    times = []
    with torch.no_grad():
        for img in test_images:
            if device.type == 'cuda':
                torch.cuda.synchronize()
            start = time.perf_counter()
            _ = model(img)
            if device.type == 'cuda':
                torch.cuda.synchronize()
            end = time.perf_counter()
            times.append((end - start) * 1000)  # ms

    avg_inference_ms = np.mean(times)

    return {
        'model_name': model_name,
        'accuracy': accuracy,
        'f1_macro': f1_macro,
        'model_size_mb': model_size_mb,
        'inference_ms': avg_inference_ms,
        'all_preds': all_preds,
        'all_labels': all_labels,
    }

In [ ]:
# Evaluar los 3 modelos
# DataLoaders para test (224x224 para ResNet y MobileNet, 299x299 para Inception)
_, _, test_loader_224 = get_dataloaders(train_transform, eval_transform)
_, _, test_loader_299 = get_dataloaders(train_transform_inception, eval_transform_inception)

results = {}

print("Evaluando ResNet50...")
results['ResNet50'] = evaluate_model(model_resnet, test_loader_224, "ResNet50", "resnet50_mango.pth")

print("Evaluando InceptionV3...")
results['InceptionV3'] = evaluate_model(model_inception, test_loader_299, "InceptionV3", "inceptionv3_mango.pth")

print("Evaluando MobileNetV2...")
results['MobileNetV2'] = evaluate_model(model_mobilenet, test_loader_224, "MobileNetV2", "mobilenetv2_mango.pth")

print("\nEvaluación completa.")

In [ ]:
# Tabla comparativa de métricas
import pandas as pd

summary = []
for name in ['ResNet50', 'InceptionV3', 'MobileNetV2']:
    r = results[name]
    summary.append({
        'Modelo': name,
        'Accuracy (%)': f"{r['accuracy']*100:.2f}",
        'F1-Score (Macro)': f"{r['f1_macro']:.4f}",
        'Tamaño (MB)': f"{r['model_size_mb']:.2f}",
        'Inferencia (ms)': f"{r['inference_ms']:.2f}",
    })

df_summary = pd.DataFrame(summary)
print("=" * 70)
print("TABLA COMPARATIVA DE MÉTRICAS")
print("=" * 70)
display(df_summary)

### Classification Report y Matrices de Confusión

In [ ]:
# Classification reports
for name in ['ResNet50', 'InceptionV3', 'MobileNetV2']:
    r = results[name]
    print(f"\n{'='*60}")
    print(f"Classification Report - {name}")
    print(f"{'='*60}")
    print(classification_report(r['all_labels'], r['all_preds'], target_names=class_names))

In [ ]:
# Matrices de confusión
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for i, name in enumerate(['ResNet50', 'InceptionV3', 'MobileNetV2']):
    r = results[name]
    cm = confusion_matrix(r['all_labels'], r['all_preds'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=axes[i])
    axes[i].set_title(f'{name}\nAcc: {r["accuracy"]*100:.1f}% | F1: {r["f1_macro"]:.3f}')
    axes[i].set_ylabel('Real')
    axes[i].set_xlabel('Predicción')
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.show()

### Gráficas comparativas de métricas

In [ ]:
# Gráficas de barras comparativas
model_names = ['ResNet50', 'InceptionV3', 'MobileNetV2']
colors = ['#2196F3', '#FF9800', '#4CAF50']

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

# Accuracy
accs = [results[m]['accuracy'] * 100 for m in model_names]
axes[0].bar(model_names, accs, color=colors)
axes[0].set_title('Accuracy (%)')
axes[0].set_ylim(0, 100)
for j, v in enumerate(accs):
    axes[0].text(j, v + 1, f'{v:.1f}%', ha='center', fontweight='bold')

# F1-Score
f1s = [results[m]['f1_macro'] for m in model_names]
axes[1].bar(model_names, f1s, color=colors)
axes[1].set_title('F1-Score (Macro)')
axes[1].set_ylim(0, 1)
for j, v in enumerate(f1s):
    axes[1].text(j, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

# Tamaño del modelo
sizes = [results[m]['model_size_mb'] for m in model_names]
axes[2].bar(model_names, sizes, color=colors)
axes[2].set_title('Tamaño del Modelo (MB)')
for j, v in enumerate(sizes):
    axes[2].text(j, v + 1, f'{v:.1f}', ha='center', fontweight='bold')

# Tiempo de inferencia
times = [results[m]['inference_ms'] for m in model_names]
axes[3].bar(model_names, times, color=colors)
axes[3].set_title('Inferencia (ms/imagen)')
for j, v in enumerate(times):
    axes[3].text(j, v + 0.5, f'{v:.1f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()